In [10]:
import pandas as pd
import numpy as np
from unidecode import unidecode
from pathlib import Path
import re
import hashlib

pd.set_option('future.no_silent_downcasting', True) # Evita avisos de downcasting


In [1]:
%run ../../config/bootstrap.py

import pandas as pd
import numpy as np
from pathlib import Path
from utils import get_project_root
project_root = get_project_root() 


# 🥉 Bronze 

Raw Data
as is production, low quality

In [3]:
path = Path(project_root) / "data/01_bronze/Medicamentos/Medicamentos.parquet"
df = pd.read_parquet(path)
df.columns

# Buscar a pasta onde está o parquet
# pasta = Path("../../data/01_bronze/Medicamentos")
# arquivo_parquet = next(pasta.glob("*.parquet"))
# df = pd.read_parquet(arquivo_parquet)

Index(['IDENTIFICACAO_NOTIFICACAO', 'RELACAO_MEDICAMENTO_EVENTO',
       'NOME_MEDICAMENTO_WHODRUG', 'PRINCIPIOS_ATIVOS_WHODRUG', 'CODIGO_ATC',
       'DETENTOR_REGISTRO', 'CONCENTRACAO', 'COMPONENTE_SUSPEITO',
       'ACAO_ADOTADA', 'PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO',
       'INDICACAO_MEDDRA', 'INDICACAO_RELATADA_NOTIFICADOR_INICIAL', 'DOSE',
       'FREQUENCIA_DOSE', 'POSOLOGIA', 'DURACAO', 'INICIO_ADMINISTRACAO',
       'FIM_ADMINISTRACAO', 'FORMA_FARMACEUTICA', 'VIA_ADMINISTRACAO',
       'VIA_ADMINISTRACAO_MAE_PAI', 'NUMELO_LOTE', 'ATUALIZADO'],
      dtype='object')

In [ ]:
df.head(2).T

# 🥈 Silver

normalized data, medium quality


In [229]:
# Normalizar as linhas
def normalize_rows(series):
    series = series.astype(str).str.strip()  # limpa espaços
    series = series.replace(["nan", "None", "NaT", "NULL"], np.nan)  # textos inválidos
    series = series.apply(lambda x: unidecode(x) if pd.notna(x) else x)
    series = series.str.upper()
    return series

for col in df.select_dtypes(include=['object']).columns:
    df[col] = normalize_rows(df[col])

In [230]:
# Converter datas no formato YYYY-MM-DD
def normalize_date_column(series):
    # Converte strings ou números para datetime
    series = pd.to_datetime(series.astype(str).str.strip(), errors="coerce", format="%Y%m%d")
    # Mantém apenas a parte da data (sem hora)
    series = series.dt.floor("d")  # arredonda para o dia
    return series

colunas_data = ["INICIO_ADMINISTRACAO", "FIM_ADMINISTRACAO"]

for col in colunas_data:
    if col in df.columns:
        df[col] = normalize_date_column(df[col])

In [231]:
def mapear_coluna(df, coluna, mapa, nome_valor=None, tipo_int64=True):
    if coluna not in df.columns:
        return df

    nome_valor = nome_valor or f"{coluna}_VALOR"
    df[nome_valor] = df[coluna]

    # Substitui pelos códigos
    df[coluna] = df[coluna].replace(mapa)
    
    if tipo_int64:
        df[coluna] = pd.to_numeric(df[coluna], errors="coerce").fillna(0).astype("Int64")

    return df

In [232]:
# VIA_ADMINISTRACAO_MAE_PAI
mapa_via = {
    # DESCONHECIDO
    "E2B R2 CODE: 065 - UNKNOWN": "DESCONHECIDO",
    "UNKNOWN": "DESCONHECIDO",
    "DESCONHECIDA": "DESCONHECIDO",
    "NAO": "DESCONHECIDO",
    "E2B R2 CODE: 050 - OTHER": "DESCONHECIDO",
    "OUTRA": "DESCONHECIDO",

    # INTRAVENOSA (IV)
    "INTRAVENOSA (NAO ESPECIFICADO)": "INTRAVENOSA (IV)",
    "VIA INTRAVENOSA": "INTRAVENOSA (IV)",
    "INFUSAO INTRAVENOSA": "INTRAVENOSA (IV)",
    "BOLUS INTRAVENOSO": "INTRAVENOSA (IV)",

    # ORAL
    "ORAL": "ORAL",
    "ORAL USE": "ORAL",
    "200 MILIGRAMA, SEMANALMENTE (QW)": "ORAL",

    # INTRAMUSCULAR (IM)
    "INTRAMUSCULAR USE": "INTRAMUSCULAR (IM)",
    "INTRAMUSCULAR": "INTRAMUSCULAR (IM)",

    # SUBCUTANEA (SC)
    "SUBCUTANEOUS": "SUBCUTANEA (SC)",
    "SUBCUTANEOUS USE": "SUBCUTANEA (SC)",
    "SUBCUTANEA": "SUBCUTANEA (SC)",
    "PARENTERAL": "SUBCUTANEA (SC)",

    # RESPIRATORIA
    "RESPIRATORY (INHALATION)": "RESPIRATORIA",

    # INTRACEREBRAL
    "INTRACEREBRAL": "INTRACEREBRAL",

    # INTRAUTERINA
    "INTRAUTERINA": "INTRAUTERINA",
}

df["VIA_ADMINISTRACAO_MAE_PAI"] = (
    df["VIA_ADMINISTRACAO_MAE_PAI"]
    .map(mapa_via)
    .fillna("DESCONHECIDO")
)

In [233]:
# RELACAO_MEDICAMENTO_EVENTO
relacao_medicamento_evento_map = {
    "SUSPEITO": 1, "CONCOMITANTE": 2, "MEDICAMENTO NAO ADMINISTRADO": 3, "INTERACAO": 4
}

# COMPONENTE_SUSPEITO
componente_suspeito_map = {
    "PRINCIPIO ATIVO": 1, "EXCIPIENTE, NAO CLASSIFICADO": 2, "SOLVENTE": 3, "CORANTE": 4,
    "CONSERVANTE": 5, "AGENTE FLAVORIZADOR": 6, "EXCESSO PERCENTUAL": 7, "ANTIOXIDANTE": 8,
    "ESTABILIZANTE": 9
}

# ACAO_ADOTADA
acao_adotada_map = {
    "RETIRADA DO MEDICAMENTO": 1, "SEM ALTERACAO DA DOSE": 2, "NAO APLICAVEL": 3,
    "REDUCAO DA DOSE": 4, "AUMENTO DA DOSE": 5
}

# VIA_ADMINISTRACAO_MAE_PAI
via_administracao_mae_pai_map = {
    "ORAL": 1, "INTRAVENOSA (IV)": 2, "SUBCUTANEA (SC)": 3, "INTRAMUSCULAR (IM)": 4,
    "INTRACEREBRAL": 5, "RESPIRATORIA": 6, "INTRAUTERINA": 7
}

df = mapear_coluna(df, "RELACAO_MEDICAMENTO_EVENTO", relacao_medicamento_evento_map)
df = mapear_coluna(df, "COMPONENTE_SUSPEITO", componente_suspeito_map)
df = mapear_coluna(df, "ACAO_ADOTADA", acao_adotada_map)
df = mapear_coluna(df, "VIA_ADMINISTRACAO_MAE_PAI", via_administracao_mae_pai_map)

In [234]:
for col in df.select_dtypes(include=['object', 'string']).columns:
    df[col] = df[col].replace(["", " ", "NAN", "NONE", "NULL"], np.nan)
    df[col] = df[col].fillna("DESCONHECIDO")

In [235]:
col = "PROBLEMAS_ADICIONAIS_RELCIONADOS_MEDICAMENTO"

df[col] = df[col].fillna("").astype(str)
df[col] = df[col].str.upper()

df[col] = df[col].str.replace(r"X000D_\|", "|", regex=True)
df[col] = df[col].str.replace(r"X000D_", "|", regex=True)
df[col] = df[col].str.replace(r"_X000D_", "|", regex=True)
df[col] = df[col].str.replace(r"X000D", "|", regex=True)

df[col] = df[col].str.replace(r"_+", "|", regex=True)
df[col] = df[col].str.replace(r"\|{2,}", "|", regex=True)
df[col] = df[col].str.replace('"', "")
df[col] = df[col].str.strip("| ")

df[col] = df[col].apply(lambda x: [i.strip() for i in x.split("|")] if x else [])

In [236]:
contagens = (
    df[col]
    .explode()
    .value_counts()
    .reset_index()
    .rename(columns={"index": "PROBLEMA", col: "QTD"})
)

In [237]:
""" excluir = [
    "DESCONHECIDO", "NAO INFORMADO", "NI", "UNKNOWN", "ASKED BUT UNKNOWN", "UNK", "DESCONHECIDA", "IGNORADO",
    "NOT AVAILABLE", "NOT PROVIDED", "DESCONHECIDO.", "-", "NAO SEI", ".", "NAO DISPONIVEL", "NAO INFORMADO.",
    "NAO IDENTIFICADO", "N/I", "ASKU"
]

df_filtrado = df[~df["NUMELO_LOTE"].isin(excluir)]

df_filtrado["NUMELO_LOTE"].value_counts(dropna=False) """

' excluir = [\n    "DESCONHECIDO", "NAO INFORMADO", "NI", "UNKNOWN", "ASKED BUT UNKNOWN", "UNK", "DESCONHECIDA", "IGNORADO",\n    "NOT AVAILABLE", "NOT PROVIDED", "DESCONHECIDO.", "-", "NAO SEI", ".", "NAO DISPONIVEL", "NAO INFORMADO.",\n    "NAO IDENTIFICADO", "N/I", "ASKU"\n]\n\ndf_filtrado = df[~df["NUMELO_LOTE"].isin(excluir)]\n\ndf_filtrado["NUMELO_LOTE"].value_counts(dropna=False) '

In [238]:
""" ## VERIFICAR

def extrair_numero(df, coluna, nome_valor=None, nome_unidade=None, tipo_valor="float"):
    if coluna not in df.columns:
        return df

    nome_valor = nome_valor or f"{coluna}_VALOR"
    nome_unidade = nome_unidade or f"{coluna}_UNIDADE"

    def extrair(valor):
        if pd.isna(valor):
            return (np.nan, np.nan)
        texto = str(valor).strip().upper()
        match = re.match(r"([\d.,]+)\s*([A-ZÇÃÉÍ]*)?", texto)
        if not match:
            return (np.nan, np.nan)
        numero_str = match.group(1)

        # Remove pontos de milhar e substitui vírgula por ponto
        numero_str = numero_str.replace(".", "").replace(",", ".")
        try:
            numero = float(numero_str) if tipo_valor == "float" else int(float(numero_str))
        except:
            numero = np.nan

        unidade = match.group(2) if match.group(2) else ""
        
        if "MG/ML" in unidade or "MGML" in unidade:
            unidade = "MG/ML"

        elif unidade in ["MG", "MILLIGRAM", "MILLIGRAM (MG)"]:
            unidade = "MILLIGRAM (MG)"

        elif unidade in ["G", "GRAM", "GRAM (G)"]:
            unidade = "GRAM (G)"

        elif unidade in ["UG", "ΜG", "MICROGRAM", "MICROGRAM (UG)", "µG"]:
            unidade = "MICROGRAM (UG)"

        elif unidade in ["%", "PERCENT", "PERCENT (%)"]:
            unidade = "PERCENT (%)"

        elif unidade in ["IU", "UI", "INTERNATIONAL UNIT", "INTERNATIONAL UNIT (IU)"]:
            unidade = "INTERNATIONAL UNIT (IU)"

        elif "ANO" in unidade:
            unidade = "ANO(S)"

        elif "MES" in unidade:
            unidade = "MES(ES)"

        elif "DIA" in unidade:
            unidade = "DIA(S)"

        elif "HORA" in unidade:
            unidade = "HORA(S)"

        elif "MINUTO" in unidade:
            unidade = "MINUTO(S)"

        elif unidade == "ML":
            unidade = "MILLILITRE (ML)"

        elif unidade == "ML":
            unidade = "MILLILITRE (ML)"
            
        return (numero, unidade)

    df[[nome_valor, nome_unidade]] = df[coluna].apply(lambda x: pd.Series(extrair(x)))

    if tipo_valor == "float":
        df[nome_valor] = df[nome_valor].astype("float64")
    else:
        df[nome_valor] = df[nome_valor].fillna(0).astype("Int64")

    return df

df = extrair_numero(df, "DOSE", tipo_valor="float") """

' ## VERIFICAR\n\ndef extrair_numero(df, coluna, nome_valor=None, nome_unidade=None, tipo_valor="float"):\n    if coluna not in df.columns:\n        return df\n\n    nome_valor = nome_valor or f"{coluna}_VALOR"\n    nome_unidade = nome_unidade or f"{coluna}_UNIDADE"\n\n    def extrair(valor):\n        if pd.isna(valor):\n            return (np.nan, np.nan)\n        texto = str(valor).strip().upper()\n        match = re.match(r"([\\d.,]+)\\s*([A-ZÇÃÉÍ]*)?", texto)\n        if not match:\n            return (np.nan, np.nan)\n        numero_str = match.group(1)\n\n        # Remove pontos de milhar e substitui vírgula por ponto\n        numero_str = numero_str.replace(".", "").replace(",", ".")\n        try:\n            numero = float(numero_str) if tipo_valor == "float" else int(float(numero_str))\n        except:\n            numero = np.nan\n\n        unidade = match.group(2) if match.group(2) else ""\n        \n        if "MG/ML" in unidade or "MGML" in unidade:\n            unidade =

In [239]:
""" ## VERIFICAR

excluir = [
    # MG
    "MG", "MILLIGRAM", "MILLIGRAM (MG)",

    # MG/ML
    "MG/ML", "MGML",

    # G
    "G", "GRAM", "GRAM (G)",

    # UG / µG
    "UG", "ΜG", "MICROGRAM", "MICROGRAM (UG)", "µG",

    # %
    "%", "PERCENT", "PERCENT (%)",

    # IU / UI
    "IU", "UI", "INTERNATIONAL UNIT", "INTERNATIONAL UNIT (IU)",

    # ANO
    "ANO", "ANOS", "ANO(S)",

    # MES
    "MES", "MESES", "MES(ES)",

    # DIA
    "DIA", "DIAS", "DIA(S)",

    # HORA
    "HORA", "HORAS", "HORA(S)",

    # MINUTO
    "MINUTO", "MINUTOS", "MINUTO(S)",

    # ML
    "ML", "MILLILITRE", "MILLILITRE (ML)",
]


df[df["DOSE_UNIDADE"].isin(excluir) == False]["DOSE_UNIDADE"] \
    .value_counts(dropna=False) \
    .reset_index() """

' ## VERIFICAR\n\nexcluir = [\n    # MG\n    "MG", "MILLIGRAM", "MILLIGRAM (MG)",\n\n    # MG/ML\n    "MG/ML", "MGML",\n\n    # G\n    "G", "GRAM", "GRAM (G)",\n\n    # UG / µG\n    "UG", "ΜG", "MICROGRAM", "MICROGRAM (UG)", "µG",\n\n    # %\n    "%", "PERCENT", "PERCENT (%)",\n\n    # IU / UI\n    "IU", "UI", "INTERNATIONAL UNIT", "INTERNATIONAL UNIT (IU)",\n\n    # ANO\n    "ANO", "ANOS", "ANO(S)",\n\n    # MES\n    "MES", "MESES", "MES(ES)",\n\n    # DIA\n    "DIA", "DIAS", "DIA(S)",\n\n    # HORA\n    "HORA", "HORAS", "HORA(S)",\n\n    # MINUTO\n    "MINUTO", "MINUTOS", "MINUTO(S)",\n\n    # ML\n    "ML", "MILLILITRE", "MILLILITRE (ML)",\n]\n\n\ndf[df["DOSE_UNIDADE"].isin(excluir) == False]["DOSE_UNIDADE"]     .value_counts(dropna=False)     .reset_index() '

In [240]:
def inferir_tipos(df):
    tipos = {}
    for col in df.columns:
        if pd.api.types.is_numeric_dtype(df[col]):
            tipos[col] = "Int64" if pd.api.types.is_integer_dtype(df[col]) else "float64"
        elif pd.api.types.is_object_dtype(df[col]):
            sample = df[col].dropna().head(20).astype(str)
            if all(pd.to_datetime(sample, format="%Y%m%d", errors="coerce").notna()):
                tipos[col] = "datetime64[ns]"
            else:
                tipos[col] = "string"
        else:
            tipos[col] = str(df[col].dtype)
    return tipos

## FORMA_FARMACEUTICA

In [4]:

df.FORMA_FARMACEUTICA.value_counts()

FORMA_FARMACEUTICA
Tablet                                                          9792
comprimido                                                      7291
Solução injetável                                               7177
solução injetável                                               7077
Solution for injection                                          6717
Comprimido                                                      4215
ampola                                                          3911
Film-coated tablet                                              3827
AMPOLA                                                          3617
Injetável                                                       3084
Concentrate for solution for infusion                           2903
Solution for infusion                                           2863
SOLUTION FOR INJECTION                                          2680
Unknown                                                         2611
COMPRIMIDO     

## VIA_ADMINISTRACAO

In [5]:
df.VIA_ADMINISTRACAO.value_counts()

VIA_ADMINISTRACAO
oral                                                            31059
infusão intravenosa                                             26376
intramuscular                                                   19866
desconhecida                                                    18764
intravenosa (não especificado)                                  18495
EV                                                              11951
Unknown                                                         11519
Oral                                                            11327
E2B R2 Code: 065 - Unknown                                       9741
subcutânea                                                       8826
Intravenosa                                                      4249
Endovenosa                                                       3855
IV                                                               3797
endovenosa                                                       3543
in

## INDICACAO_MEDDRA

In [8]:
dim_indicacao_meddra = df.INDICACAO_MEDDRA.value_counts().reset_index().rename(columns={"index": "INDICACAO_MEDDRA", "INDICACAO_MEDDRA": "QTD"})
dim_indicacao_meddra.head(10)

,QTD,count
0,Uso de medicamento para indicação desconhecida,19595
1,Produto usado para indicação desconhecida,16515
2,Doença de Crohn,6875
3,Artrite reumatoide,6207
4,Hipertensão,4399
5,Profilaxia de COVID-19,4179
6,Esclerose múltipla,4085
7,Mieloma múltiplo,3449
8,Espondilite anquilosante,3139
9,Câncer de mama,3119


# 🥇 Gold


In [241]:
# Caminho do arquivo Gold
caminho_gold = "../data/03_gold/dim_medicamentos/dim_medicamentos.parquet"
df.to_parquet(caminho_gold, index=False)